In [ ]:
import torch
import torch.nn as nn

# Motivation
## RNN limitations
- Token-by-token computation (can't utilize parallel)
- Far information passed through many recurrent steps
- Optimization over long contexts is unstable/inefficient
## CNN limitations
- Parallel convolution but local receptive field (limited context)
- Long-range context capturing requires deeper stacks/larger kernels
- Context expansion is indirect (compare to token-token interaction)
## Key idea
Each step, each token can see all tokens
# Shape
## Notation
- d_k, d_v: per-head key, val dimensions
- d_ff: FFN hidden expansion dimension
- Internal projection may change
- B x n x d_model unchanged entire Transformer
## Flow
- X -> Q, K, V
- (B, n, d_model) -> (B, n, d_k)
- QK^T -> (B, n, n)
- AV -> (B, n, d_v)
- Output -> (B, n, d_model)
- Multihead: add h right after B and formula do not change
# Attention
## Scaled dot-product attention
- A = softmax(QK^T/sqrt(d_k))
- Y = AV
- Shape: A -> (B, n, n)
- Row i of A is a distribution over tokens attended by token i
## Attention as weighted average
- y_i = sum(alpha_ij v_j)
- alpha_i is a distribution -> alpha_ij >= 0, sum(alpha_i) = 1.
- alpha_ij measures how much token i attends to token j
- y_i lies in the convex hull of {v_1, v2, ..., v_n}
- Attention mixes values (V = XW_V), not raw input tokens X
- We only reweighting values, not creating new information
- Therefore attention = content-dependent weighted averaging over tokens
## Limitation
- Only recombines existing value vectors
- Can not create new features beyond linear mixtures
- Representation power relies heavily on: 
+ value projection W_v (select information)
+ feed-forward network FFN (create new representations)
# Masking
- IMPORTANT: ignore padded positions (no semantic), enforce causality in autoregressive decoding (no future leakage)
- A = softmax(QK^T/sqrt(d_k) + M)
- M_ij = -INF => A_ij = 0
- Padding mask: hide PAD tokens
- Causal mask: hide future positions
- Shape: (B, 1, n, n) broadcast over all heads, (B, n, n) direct masking
- Values: 0 = allowed, -INF = blocked
- IMPORTANT#2: Apply mask BEFORE softmax
# Multihead attention
X -> Projection QKV -> Split into h heads -> Scaled dot product attention each head -> Concat head -> Output projection W_O -> Y
- MHA(X) = Concat(head_1, head_2, ..., head_n) W_O
- d_K = d_V = d_model / h
- Different heads specialize in different relation patterns (local, global, syntax-like, delimiter-focused)
- W_O is added to mix projection (Single head do not need extra combination as output already in the dimension of transformer)
## Parameter count
- W_Q: token -> query: `d_model x d_model`
- W_K: token -> key: `d_model x d_model`
- W_V: token -> value: `d_model x d_model`
- W_O: concat -> d_model: `d_model x d_model`
- Without bias: `4d_model ^2`
- With bias: `4d_model ^2 + 4d_model`
# Transformer block:
X -> MultiHead Attention -> Skip connection -> LayerNorm -> FFN -> Skip connection -> LayerNorm -> Y
- MHA: Mixes information accross token positions -> Each token can read global context
- LayerNorm: Stabilizes activation scale, optimization on deep stack
- FFN: transforms channels per token (expand -> nonlinearity -> project), increase representation capacity
- Skip connection: preserves identity signal and improves gradient flow through deep network
- Alternate token mixing (MHA) and channel mixing (FFN) + normalization + residual paths for stable training
- x -> Sublayer -> +x -> LayerNorm (traditional)
- Less stable to pre-norm without careful training setup
## Pre-norm computation graph
X -> LayerNorm -> MultiHead Attention -> Skip connection -> LayerNorm -> FFN -> Skip connection -> Y
- x -> LayerNorm -> Sublayer -> +x 
- Easier to optimize in deeper stack
## Feed-Forward Network
x -> W1 (expand to d_ff) -> W2 (project back to model) -> x'
- W1: d_model -> d_ff: `d_model x d_ff`
- W2: d_ff -> d_model: `d_ff x d_model`
- Without bias: `2d_model d_ff`
- With bias: `2d_model d_ff + d_ff + d_model`
- Common: d_ff approx. 4d_model -> Parameters: `8d_model ^2` (no bias)
## Total parameter:
- MHA: `4d_model ^2` (`4d_model ^2 + 4d_model`)
- FFN: `2d_model d_ff` (`2d_model d_ff + d_ff + d_model`)
- LayerNorm: `2d_model`
- Total: `4d_model ^2 + 2d_model d_ff` (`4d_model ^2 + 2d_model d_ff + d_ff + 5d_model`)
- Mostly linear projections, softmax and masking has no learnable parameters
## Computation complexity
- Q, K, V: O(nd^2)
- Attention: O(dn^2)
- FFN: O(ndd_ff) approx. O(4nd^2) with d_ff = 4d
- Large n => Attention costly (cost go to attn when learn long sequence)
- Large d/d_ff => FFN/ projection cosly (cost go to projection if model is rich in representation)
## Compare with CNN
- CNN = bias + efficiency:
+ Strong inductive bias (locality, translation invariance)
+ Efficient but less flexible
- Attention = flexibility + data dependency:
+ Weak inductive bias
+ Learns data relationships directly
## Complexity:
- Memory bottleneck: matrix size n x n (per head, per batch)
- Compute bottleneck: quadratic interaction O(dn^2)
- Sequence length is often the dominant bottleneck


In [ ]:
# Minimal attention
score = (Q @ K.transpose(-2, -1)) / (d ** 0.5)
score = score + mask
A = torch.softmax(score, dim=-1)
y = A @ V
# Shape:
# scores: (B, n, n)
# A: (B, n, n)
# y: (B, n, d_v)

In [ ]:
# API MHA
multihead = nn.MultiheadAttention(d_model, nhead)
# One encoder block (MHA + FFN + norm + dropout)
encoder = nn.TransformerEncoderLayer(d_model, nhead)
# Encoder stack
encoder_stack = nn.TransformerEncoder(encoder, num_layers)
# Watch out: sequence first unless batch_first=True

In [ ]:
# MHSA
mhsa = nn.MultiheadAttention(embed_dim=512, num_heads=8, batch_first=True)

x = torch.rand(4, 128, 512)  # (B, n, d_model)
y, attn = mhsa(x, x, x, need_weights=True)  # (B, n, d_model), (B, n, n)

# Encoder layer + stacked encoder
encoder_layer = nn.TransformerEncoderLayer(d_model=512, nhead=8, dim_feedforward=2048, batch_first=True)
encoder_stack = nn.TransformerEncoder(encoder_layer, num_layers=6)
z = encoder_stack(x)  # (B, n, d_model)

# Engineering insight
- Cost runtime is dominated by attention activation tensors
- Parameter count is dominated by FFN projections
- Throughput depends strongly on batch size, seq length, kernel implementation
- For development, optimize sequence length and memory movement before only scaling model width